# Notebook 06: Transformer (Audio Spectrogram Transformer)
This notebook finetunes a pretrained Audio Spectrogram Transformer (AST) on the Logical Access dataset.

In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from src.models.ast import AudioSpectrogramTransformer
from src.models import Trainer

print("Transformer packages imported successfully!")


Transformer packages imported successfully!


In [2]:
# Load dataset
X = np.load('datasets/processed/X_features.npy')
y = np.load('datasets/processed/y.npy')

# Use first 64 feature dimensions (representing Mel-spectrogram coefficients)
X_mel = X[:, :, :64, :]

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_mel, y, test_size=0.2, random_state=42)

train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)
print("DataLoader initialized.")


DataLoader initialized.


In [3]:
# Train AST
ast_best_path = 'models/ast_best.pt'
ast_model = AudioSpectrogramTransformer(num_classes=2, pretrained=False)
if os.path.exists(ast_best_path):
    print('AST checkpoint found. Skipping training.')
    ast_model.load_model(ast_best_path)
elif os.path.exists('models/best_model.pt') and os.path.getsize('models/best_model.pt') > 1000000000:
    print('Trained AST checkpoint found as best_model.pt. Renaming and skipping training.')
    if os.path.exists(ast_best_path): os.remove(ast_best_path)
    os.rename('models/best_model.pt', ast_best_path)
    ast_model.load_model(ast_best_path)
else:
    print('Training AST...')
    optimizer = torch.optim.AdamW(ast_model.parameters(), lr=5e-5, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    trainer = Trainer(
        model=ast_model,
        optimizer=optimizer,
        criterion=criterion,
        device='cpu',
        mixed_precision=False,
        early_stopping_patience=2
    )
    ast_history = trainer.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=2,
        checkpoint_dir='models'
    )
    if os.path.exists('models/best_model.pt'):
        if os.path.exists(ast_best_path): os.remove(ast_best_path)
        os.rename('models/best_model.pt', ast_best_path)
        print('AST model successfully finetuned and saved as ast_best.pt!')


AST checkpoint found. Skipping training.
